# Segmentation des vins (Wine Quality) : K-means et CAH

> ### 📌 Source du jeu de données
> **Wine Quality** : UCI Machine Learning Repository.
> - Lien : https://archive.ics.uci.edu/dataset/186/wine+quality
> - Citation : Cortez, P., Cerdeira, A., Almeida, F., Matos, T., & Reis, J. (2009). *Wine Quality* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C56S3T
> - Licence : Creative Commons Attribution 4.0 (CC BY 4.0)

Ce jeu de données contient des mesures physico-chimiques d'échantillons de vins rouges et blancs *vinho verde* du nord du Portugal. L'objectif ici est de faire une **segmentation (clustering)** des vins pour identifier des groupes homogènes, avec **K-means** puis la **CAH**, puis de déployer le meilleur modèle dans une application Streamlit.

#### Description des variables

| Variable | Type | Description |
|----------|------|-------------|
| `fixed_acidity` | Numérique | Acidité fixe (acides non volatils du vin). |
| `volatile_acidity` | Numérique | Acidité volatile (liée à l'acide acétique). |
| `citric_acid` | Numérique | Acide citrique. |
| `residual_sugar` | Numérique | Sucre résiduel après fermentation. |
| `chlorides` | Numérique | Teneur en chlorures (sel). |
| `free_sulfur_dioxide` | Numérique | Dioxyde de soufre libre. |
| `total_sulfur_dioxide` | Numérique | Dioxyde de soufre total. |
| `density` | Numérique | Densité du vin. |
| `pH` | Numérique | Niveau d'acidité (pH). |
| `sulphates` | Numérique | Sulfates. |
| `alcohol` | Numérique | Degré d'alcool. |
| `quality` | Numérique | Note de qualité (0 à 10) donnée par des dégustateurs. **Non utilisée pour le clustering** : elle sert seulement à interpréter les classes. |
| `color` | Catégorielle | Couleur du vin (rouge ou blanc). **Non utilisée pour le clustering** : elle sert seulement à interpréter les classes. |

### <font color='gold'> Import des librairies requises

In [ ]:
# Installer les packages (Colab)
%pip install -q ucimlrepo

# Importer packages
import pandas as pd
import numpy as np
import plotly.express as px

# Supprimer les warnings
import warnings
warnings.filterwarnings('ignore')

### <font color='gold'> Importer et lire les données

In [ ]:
# Importer les données directement depuis UCI (id = 186 : Wine Quality)
from ucimlrepo import fetch_ucirepo
wine = fetch_ucirepo(id=186)

# Données complètes (variables + qualité + couleur)
brut = wine.data.original.copy()

# Choix des vins : 'rouge', 'blanc' ou 'tous'
CHOIX = 'tous'
if CHOIX != 'tous' and 'color' in brut.columns:
    brut = brut[brut['color'] == {'rouge': 'red', 'blanc': 'white'}[CHOIX]]

# Les 11 variables physico-chimiques utilisées pour le clustering
colonnes = ['fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides',
            'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'pH', 'sulphates', 'alcohol']
df = brut[colonnes].copy()
quality = brut['quality'].copy()
couleur = brut['color'].copy() if 'color' in brut.columns else None

In [ ]:
# Afficher les données
df.head()

In [ ]:
# Dimension des données
df.shape

In [ ]:
# Infos des données
df.info()

In [ ]:
# Décrire les données
df.describe().T

### <font color='gold'> Nettoyage des données

In [ ]:
# 1) Valeurs manquantes
print("Valeurs manquantes par variable :")
print(df.isna().sum())

# 2) Doublons
print("\nNombre de lignes dupliquées :", df.duplicated().sum())

In [ ]:
# Supprimer les valeurs manquantes (s'il y en a) et les doublons
masque = df.notna().all(axis=1) & ~df.duplicated()
df = df[masque]
quality = quality[masque]
if couleur is not None:
    couleur = couleur[masque]
print("Dimension après nettoyage :", df.shape)

Les variables n'ont pas la même échelle (par exemple `total_sulfur_dioxide` est bien plus grand que `chlorides`) : on les **standardise** avant le clustering pour qu'aucune ne domine les distances.

### <font color='gold'> Préparation des données

In [ ]:
# Renommer les index
df.index = ['Vin' + str(i + 1) for i in range(len(df))]
quality.index = df.index
if couleur is not None:
    couleur.index = df.index
df.head()

In [ ]:
# Standardisation (moyenne 0, écart-type 1)
from sklearn.preprocessing import StandardScaler
preprocess = StandardScaler()
x = preprocess.fit_transform(df)
x[:5]

In [ ]:
# Importer PCA
from sklearn.decomposition import PCA
# Instancier le modèle PCA avec n_components = 2
pca_model = PCA(n_components=2)
# Matrice des 2 composantes
feature_pca = pca_model.fit_transform(x)
# Convertir en dataframe
New_df = pd.DataFrame(feature_pca, columns=['PC1', 'PC2'])

In [ ]:
# Part de variance des 2 composantes
np.round(sum(pca_model.explained_variance_ratio_) * 100, 2)

### <font color='gold'> K-means

In [ ]:
# Importer KMeans et la silhouette
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Méthode du coude (inertie) et indice de silhouette pour chaque valeur de k
resultats = []
for k in range(2, 9):
    modele = KMeans(n_clusters=k, init='random', max_iter=500, n_init=10, random_state=42)
    part = modele.fit_predict(x)
    resultats.append({'k': k, 'inertie': modele.inertia_, 'silhouette': silhouette_score(x, part)})
sil_par_k = pd.DataFrame(resultats)
sil_par_k

In [ ]:
# Méthode du coude : on cherche le k où la courbe de l'inertie « plie »
px.line(sil_par_k, x='k', y='inertie', markers=True, title="Méthode du coude (inertie)").show()

# Indice de silhouette selon k : plus il est élevé, mieux les classes sont séparées
px.line(sil_par_k, x='k', y='silhouette', markers=True, title="Silhouette selon k").show()

In [ ]:
# k retenu : meilleur indice de silhouette entre 3 et 6 groupes.
# (La silhouette favorise souvent k = 2 ; on impose au moins 3 groupes pour une segmentation utile.)
# Tu peux remplacer cette ligne par une valeur choisie d'après le coude, ex : k_kmeans = 4
k_kmeans = int(sil_par_k[sil_par_k['k'].between(3, 6)].sort_values('silhouette').iloc[-1]['k'])
print("k retenu pour K-means :", k_kmeans)

# Instancier le modèle
kmeans_model = KMeans(n_clusters=k_kmeans, init='random', max_iter=500, n_init=10, random_state=42)
# Entraîner le modèle
kmeans_model.fit(x)

#### <font color='teal'> Prédiction

In [ ]:
# Prédiction de la partition des données
kmeans_partition = kmeans_model.predict(x)
# Afficher le début de la partition
kmeans_partition[:100]

In [ ]:
# Nombre d'observations dans chaque classe
val = pd.Series(kmeans_partition).value_counts().sort_index()
val

In [ ]:
# Fonction de visualisation du diagramme en barres des effectifs
def bar_plot(val):
    fig = px.bar(x=val.index.astype(str), y=val.values, color=val.index.astype(str), text_auto=True,
                 labels={'x': 'Classe', 'y': 'Effectif', 'color': 'Classe'})
    fig.show()

# Fonction de visualisation des classes sur les 2 premières composantes
def scatter_plot(New_df, Pred):
    fig = px.scatter(New_df, x='PC1', y='PC2', color=Pred.astype(str), labels={'color': 'Classe'})
    fig.show()

# Effectifs et classes du K-means
bar_plot(val)
scatter_plot(New_df, kmeans_partition)

##### <font color='teal'> Indice de silhouette de la partition du K-means

In [ ]:
# Silhouette score du K-means
sil_kmeans = silhouette_score(x, kmeans_partition)
print(f"Silhouette score K-means : {sil_kmeans:.4f}")

### <font color='gold'> CAH

In [ ]:
# Importer linkage, dendrogram et fcluster
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
import matplotlib.pyplot as plt

# Matrice des liaisons (critère de Ward, adapté à des données standardisées)
Z = linkage(x, 'ward')

# Dendrogramme tronqué aux 30 dernières fusions (lisible même avec beaucoup d'individus)
plt.figure(figsize=(14, 6))
dendrogram(Z, truncate_mode='lastp', p=30, leaf_rotation=90., leaf_font_size=10, show_contracted=True)
plt.title("Dendrogramme (CAH, critère de Ward)")
plt.xlabel("Classes regroupées (effectif entre parenthèses)")
plt.ylabel("Distance")
plt.show()

In [ ]:
# Indice de silhouette de la CAH pour chaque nombre de classes
resultats_cah = []
for k in range(2, 9):
    part = fcluster(Z, k, criterion='maxclust') - 1
    resultats_cah.append({'k': k, 'silhouette': silhouette_score(x, part)})
sil_cah_par_k = pd.DataFrame(resultats_cah)
px.line(sil_cah_par_k, x='k', y='silhouette', markers=True).show()
sil_cah_par_k

In [ ]:
# Nombre de classes retenu : meilleur indice de silhouette entre 3 et 6 classes
# (à remplacer par une valeur lue sur le dendrogramme si besoin, ex : k_cah = 3)
k_cah = int(sil_cah_par_k[sil_cah_par_k['k'].between(3, 6)].sort_values('silhouette').iloc[-1]['k'])
print("Nombre de classes retenu pour la CAH :", k_cah)

from sklearn.cluster import AgglomerativeClustering
# Initier AgglomerativeClustering
cah_model = AgglomerativeClustering(n_clusters=k_cah, linkage='ward')
# Prédire la partition des données
cah_partition = cah_model.fit_predict(x)
cah_partition[:100]

In [ ]:
# Nombre d'observations dans chaque classe
val1 = pd.Series(cah_partition).value_counts().sort_index()
print(val1)

# Diagramme en barres et visualisation des classes
bar_plot(val1)
scatter_plot(New_df, cah_partition)

# Silhouette score
sil_cah = silhouette_score(x, cah_partition)
print(f"Silhouette score CAH : {sil_cah:.4f}")

### <font color='gold'> Comparaison K-means / CAH

In [ ]:
# Comparaison des silhouettes
Model_Names = ['KMEANS', 'CAH']
sil = [sil_kmeans, sil_cah]
fig = px.bar(x=Model_Names, y=sil, color=Model_Names, text_auto='.4f',
             labels={'x': 'Modèle', 'y': 'Silhouette score'})
fig.show()

In [ ]:
# Sélection automatique du meilleur modèle (silhouette la plus élevée)
if sil_kmeans >= sil_cah:
    meilleur_nom = 'KMEANS'
    partition_finale = kmeans_partition
else:
    meilleur_nom = 'CAH'
    partition_finale = cah_partition
print("Meilleur modèle :", meilleur_nom)

### <font color='gold'> Déploiement du meilleur modèle

In [ ]:
from scipy.spatial.distance import cdist

# Centroïdes des classes du meilleur modèle (dans l'espace transformé)
classes = np.unique(partition_finale)
centroides = np.array([x[partition_finale == c].mean(axis=0) for c in classes])

# Prédiction : la classe du centroïde le plus proche
# `nouvel_individu` = liste des valeurs, dans l'ordre des colonnes de `colonnes`
def predire_classe(nouvel_individu):
    ligne = pd.DataFrame([nouvel_individu], columns=colonnes)
    z = preprocess.transform(ligne)
    return classes[cdist(z, centroides).argmin()]

In [ ]:
# Test rapide
print("Classe prédite          :", predire_classe(df.iloc[0].tolist()))
print("Classe réelle du modèle :", partition_finale[0])

#### <font color='teal'> Interpréter les classes

Avant de livrer l'application, il faut savoir **ce que représente chaque classe**, pour afficher un message compréhensible. On regarde le profil moyen de chaque classe, ainsi que la qualité moyenne et la part de vins rouges (variables **non utilisées** pour construire les classes).

In [ ]:
# Profil moyen de chaque classe
profil = df.groupby(partition_finale).mean().round(2)
profil['effectif'] = pd.Series(partition_finale).value_counts().sort_index().values
profil['qualité_moyenne'] = quality.groupby(partition_finale).mean().round(2).values
if couleur is not None:
    profil['% rouge'] = (couleur.eq('red').groupby(partition_finale).mean() * 100).round(1).values
profil

**À faire :** lis le tableau ci-dessus (alcool, acidité, sucre, soufre, qualité, couleur...) et remplace les noms par défaut par un nom qui décrit vraiment chaque groupe (ex : « vins blancs doux », « vins rouges très acides »...).

In [ ]:
# Noms des classes : à personnaliser d'après le profil ci-dessus
noms_classes = {int(c): f"Groupe {c}" for c in classes}
noms_classes

In [ ]:
# Exemple de message renvoyé par l'application
def affecter_vin(*valeurs):
    classe = predire_classe(list(valeurs))
    return "Ce vin appartient au groupe : " + noms_classes[int(classe)]

affecter_vin(*df.iloc[0].tolist())

### <font color='gold'> Sauvegarde du modèle pour l'application Streamlit

In [ ]:
import joblib

# On sauvegarde tout ce dont l'application Streamlit a besoin
bundle = {
    'nom_modele': meilleur_nom,
    'source': 'https://archive.ics.uci.edu/dataset/186/wine+quality (Cortez et al., 2009, https://doi.org/10.24432/C56S3T)',
    'colonnes': colonnes,
    'preprocess': preprocess,
    'centroides': centroides,
    'classes': classes,
    'noms_classes': noms_classes,
    'minimum': df.min().to_dict(),
    'maximum': df.max().to_dict(),
    'mediane': df.median().to_dict(),
}
joblib.dump(bundle, 'wine_clustering.joblib')
print("Modèle sauvegardé dans wine_clustering.joblib")

# Dans Colab, pour récupérer le fichier :
# from google.colab import files
# files.download('wine_clustering.joblib')

#### <font color='teal'> Utilisation dans l'application Streamlit

Dans l'appli, il suffit de charger ce fichier et de reprendre la même logique :

```python
import joblib, pandas as pd
from scipy.spatial.distance import cdist

m = joblib.load("wine_clustering.joblib")
ligne = pd.DataFrame([valeurs_saisies], columns=m["colonnes"])
z = m["preprocess"].transform(ligne)
classe = m["classes"][cdist(z, m["centroides"]).argmin()]
print(m["noms_classes"][classe])
```